# DT–SDG Cluster 3–16: Digital Transformation across Health (SDG 3) and Institutions (SDG 16)

Corpus-based computational pipeline mapping digital transformation discourse across the
SDG 3 (health) and SDG 16 (peace, justice, strong institutions) cluster.

**Method.** SentenceTransformer embeddings (`all-MiniLM-L6-v2`), cosine similarity,
network analysis (eigenvector centrality, core–periphery decomposition, Louvain community
detection), and theory-conditioned structural analysis. `SEED = 42` throughout for
reproducibility.

**Usage.** Place the source `.docx` corpus in `data/files/` (or set `DATA_DIR`).
Outputs (CSVs, figures) are written to `outputs/` (or `OUTPUT_DIR`).
Run cells top to bottom.


In [ ]:
# =========================
# SETUP
# =========================
# Portable configuration. Set DATA_DIR / OUTPUT_DIR via environment variables,
# or edit the defaults below. Drop the source .docx corpus into DATA_DIR.
#
# Google Colab users: uncomment the two lines below to mount Drive, then point
# BASE_PATH / RESULT_PATH at your Drive folders.
# from google.colab import drive
# drive.mount('/content/drive')

import os, re, random
import numpy as np
import pandas as pd

from docx import Document
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# Program-wide reproducibility standard
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Paths (override with env vars; defaults are repo-relative) ---
BASE_PATH   = os.environ.get("DATA_DIR",   os.path.join("data", "files"))
RESULT_PATH = os.environ.get("OUTPUT_DIR", os.path.join("outputs"))
os.makedirs(RESULT_PATH, exist_ok=True)

print("BASE_PATH  :", BASE_PATH)
print("RESULT_PATH:", RESULT_PATH)


In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print("Model loaded")

In [ ]:
def extract_structured_text(doc):
    text = []
    capture = False

    for p in doc.paragraphs:
        t = p.text.strip().lower()

        if "abstract" in t:
            capture = True

        elif "introduction" in t:
            capture = True

        elif ("conclusion" in t) or ("discussion" in t):
            capture = True

        elif any(x in t for x in ["references", "appendix", "acknowledgment"]):
            capture = False

        if capture:
            text.append(t)

    return " ".join(text)

In [ ]:
records = []

for root, _, files in os.walk(BASE_PATH):
    for f in files:
        if f.lower().endswith(".docx"):
            path = os.path.join(root, f)

            try:
                doc = Document(path)
                text = extract_structured_text(doc)

                if len(text.strip()) > 300:
                    records.append({
                        "paper_id": f.replace(".docx", ""),
                        "text": text
                    })

            except:
                continue

df_documents = pd.DataFrame(records)

df_documents.to_csv(os.path.join(RESULT_PATH, "df_documents.csv"), index=False)

print("Corpus:", df_documents.shape)

In [ ]:
df_documents.to_csv(os.path.join(RESULT_PATH, "df_documents.csv"), index=False)
#df_theory.to_csv(os.path.join(RESULT_PATH, "df_theory.csv"), index=False)
#df_sdg.to_csv(os.path.join(RESULT_PATH, "df_sdg.csv"), index=False)
#df_master.to_csv(os.path.join(RESULT_PATH, "df_master.csv"), index=False)

In [ ]:
DT_THEORY_TEXTS = {
    "TAM": "technology acceptance model usefulness ease adoption",
    "UTAUT": "technology use performance expectancy effort expectancy",
    "STS": "socio technical systems human technology interaction",
    "TOE": "technology organization environment adoption framework",
    "RBV": "resource based view capabilities advantage",
    "Dynamic_Capabilities": "sensing seizing transforming capabilities",
    "IDT": "innovation diffusion adoption process",
    "Stakeholder_Theory": "stakeholder governance multi actor",
    "Just_Digital": "digital justice fairness inclusion equity"
}

theory_labels = list(DT_THEORY_TEXTS.keys())

theory_embeddings = model.encode(
    list(DT_THEORY_TEXTS.values()),
    normalize_embeddings=True
)

doc_embeddings = model.encode(
    df_documents["text"].tolist(),
    normalize_embeddings=True
)

sim = cosine_similarity(doc_embeddings, theory_embeddings)

df_theory = pd.DataFrame(sim, columns=theory_labels)
df_theory.insert(0, "paper_id", df_documents["paper_id"])

df_theory["dominant_dt_theory"] = df_theory[theory_labels].idxmax(axis=1)

df_theory.to_csv(os.path.join(RESULT_PATH, "dt_theory_similarity_matrix.csv"), index=False)

print("Theory done")

In [ ]:
SDG_TEXTS = {
    "SDG3": """
    health healthcare public health wellbeing medical services disease prevention
    digital health telemedicine e-health healthcare systems patient outcomes
    access to healthcare health inequality epidemiology mental health
    healthcare delivery health infrastructure data-driven healthcare
    """,

    "SDG16": """
    governance institutions policy regulation transparency accountability justice
    rule of law digital governance e-government public administration
    regulatory frameworks compliance institutional capacity trust government
    data governance digital policy decision-making authority systems
    """
}
sdg_labels = list(SDG_TEXTS.keys())

sdg_embeddings = model.encode(
    list(SDG_TEXTS.values()),
    normalize_embeddings=True
)

sim_sdg = cosine_similarity(doc_embeddings, sdg_embeddings)

df_sdg = pd.DataFrame(sim_sdg, columns=sdg_labels)
df_sdg.insert(0, "paper_id", df_documents["paper_id"])

df_sdg["dominant_sdg"] = df_sdg[sdg_labels].idxmax(axis=1)

df_sdg.to_csv(os.path.join(RESULT_PATH, "sdg_similarity_matrix.csv"), index=False)

print("SDG done")

In [ ]:
df_master = df_theory.merge(
    df_sdg,
    on="paper_id",
    how="inner"
)

df_master.to_csv(os.path.join(RESULT_PATH, "df_master.csv"), index=False)

print("Master:", df_master.shape)
df_master.to_csv(os.path.join(RESULT_PATH, "df_master.csv"), index=False)


In [ ]:
THRESHOLD = 0.30
edges = []

for _, row in df_theory.iterrows():
    active = [t for t in theory_labels if row[t] >= THRESHOLD]
    for i in range(len(active)):
        for j in range(i+1, len(active)):
            edges.append((active[i], active[j]))

df_edges = pd.DataFrame(edges, columns=["A", "B"]).value_counts().reset_index(name="weight")

G = nx.Graph()

for _, r in df_edges.iterrows():
    G.add_edge(r["A"], r["B"], weight=r["weight"])

centrality = nx.eigenvector_centrality(G, max_iter=1000)

df_cent = pd.DataFrame({
    "theory": list(centrality.keys()),
    "score": list(centrality.values())
}).sort_values("score", ascending=False)

df_cent.to_csv(os.path.join(RESULT_PATH, "theory_centrality.csv"), index=False)

print("Network done")

In [ ]:
pos = nx.spring_layout(G, seed=42)

sizes = [3000 * centrality[n] for n in G.nodes()]

plt.figure(figsize=(10,10))
nx.draw_networkx(G, pos, node_size=sizes, with_labels=True, alpha=0.8)
plt.title("Theory Network")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# SECTION — DESCRIPTIVE STATISTICS (EXTENDED)
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# -----------------------------
# Load data
# -----------------------------
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))

# -----------------------------
# 1. DT distribution
# -----------------------------
desc_dt = (
    df_master["dominant_dt_theory"]
    .value_counts()
    .reset_index()
)

desc_dt.columns = ["DT_Theory", "Count"]
desc_dt["Percent"] = (
    desc_dt["Count"] / desc_dt["Count"].sum() * 100
).round(2)

desc_dt.to_csv(os.path.join(RESULT_PATH, "desc_dt_overall.csv"), index=False)

# -----------------------------
# 2. SDG distribution
# -----------------------------
desc_sdg = (
    df_master["dominant_sdg"]
    .value_counts()
    .reset_index()
)

desc_sdg.columns = ["SDG", "Count"]
desc_sdg["Percent"] = (
    desc_sdg["Count"] / desc_sdg["Count"].sum() * 100
).round(2)

desc_sdg.to_csv(os.path.join(RESULT_PATH, "desc_sdg_overall.csv"), index=False)

# -----------------------------
# 3. DT × SDG cross-tab
# -----------------------------
dt_sdg_counts = pd.crosstab(
    df_master["dominant_dt_theory"],
    df_master["dominant_sdg"]
)

dt_sdg_counts.to_csv(os.path.join(RESULT_PATH, "dt_by_sdg_counts.csv"))

dt_sdg_percent = (
    dt_sdg_counts.div(dt_sdg_counts.sum(axis=0), axis=1)
    * 100
).round(2)

dt_sdg_percent.to_csv(os.path.join(RESULT_PATH, "dt_by_sdg_percent.csv"))

# ============================================================
# NEW PART — PUBLICATION YEAR EXTRACTION
# ============================================================

# -----------------------------
# Extract year from text
# -----------------------------
def extract_year(text):
    if not isinstance(text, str):
        return np.nan

    years = re.findall(r"(20\d{2})", text)
    if years:
        years = [int(y) for y in years if 2010 <= int(y) <= 2026]
        if years:
            return min(years)  # first plausible year
    return np.nan

df_documents["year"] = df_documents["text"].apply(extract_year)

# Merge year into master
df_year = df_documents[["paper_id", "year"]]
df_master = df_master.merge(df_year, on="paper_id", how="left")

df_master.to_csv(os.path.join(RESULT_PATH, "df_master_with_year.csv"), index=False)

# -----------------------------
# Year distribution table
# -----------------------------
year_dist = (
    df_master["year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

year_dist.columns = ["Year", "Count"]

year_dist.to_csv(os.path.join(RESULT_PATH, "year_distribution.csv"), index=False)

# ============================================================
# FIGURE 1 — PUBLICATION TREND (LINE)
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(year_dist["Year"], year_dist["Count"], marker="o")
plt.title("Publication Trend Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Papers")
plt.grid(True)
plt.tight_layout()

plt.savefig(os.path.join(RESULT_PATH, "Figure_Publication_Trend.png"), dpi=300)
plt.show()

# ============================================================
# FIGURE 2 — HISTOGRAM (YEAR DISTRIBUTION)
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(df_master["year"].dropna(), bins=15)
plt.title("Histogram of Publication Years")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig(os.path.join(RESULT_PATH, "Figure_Year_Histogram.png"), dpi=300)
plt.show()

# ============================================================
# FIGURE 3 — SDG DISTRIBUTION
# ============================================================

plt.figure(figsize=(6,4))
plt.bar(desc_sdg["SDG"], desc_sdg["Count"])
plt.title("SDG Distribution")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(os.path.join(RESULT_PATH, "Figure_SDG_Distribution.png"), dpi=300)
plt.show()

# ============================================================
# FIGURE 4 — DT THEORY DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))
plt.barh(desc_dt["DT_Theory"], desc_dt["Count"])
plt.gca().invert_yaxis()
plt.title("DT Theory Distribution")
plt.tight_layout()

plt.savefig(os.path.join(RESULT_PATH, "Figure_DT_Distribution.png"), dpi=300)
plt.show()

# ============================================================
# SUMMARY PRINT
# ============================================================

print("DESCRIPTIVE STATISTICS COMPLETE")
print("Files created:")
print("- desc_dt_overall.csv")
print("- desc_sdg_overall.csv")
print("- dt_by_sdg_counts.csv")
print("- dt_by_sdg_percent.csv")
print("- year_distribution.csv")
print("- df_master_with_year.csv")
print("- Figures saved")

In [ ]:
# ============================================================
# FULL TABLE GENERATION (ROBUST, SELF-CONTAINED, VALIDATED)
# ============================================================

import os
import pandas as pd
import numpy as np
import re
import itertools
import networkx as nx

# -----------------------------
# LOAD CORE DATA
# -----------------------------
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))

# -----------------------------
# VALIDATION (NON-NEGOTIABLE)
# -----------------------------
assert df_master.shape[0] > 0, "❌ df_master empty"
assert df_documents.shape[0] > 0, "❌ df_documents empty"

required_cols = ["paper_id", "dominant_dt_theory", "dominant_sdg"]
for c in required_cols:
    assert c in df_master.columns, f"❌ Missing column: {c}"

if "sdg_margin" not in df_master.columns:
    print("⚠ sdg_margin missing → creating fallback (0)")
    df_master["sdg_margin"] = 0


# ============================================================
# TABLE 1 — CORPUS OVERVIEW
# ============================================================
table1 = pd.DataFrame({
    "Measure": ["Total documents", "Variables"],
    "Value": [df_documents.shape[0], df_documents.shape[1]]
})

table1.to_csv(os.path.join(RESULT_PATH, "Table_1_Corpus.csv"), index=False)


# ============================================================
# TABLE 4 — DT DISTRIBUTION
# ============================================================
table4 = (
    df_master["dominant_dt_theory"]
    .value_counts()
    .reset_index()
)

table4.columns = ["DT Theory", "Frequency"]
table4["Percentage (%)"] = (
    table4["Frequency"] / table4["Frequency"].sum() * 100
).round(1)

table4.to_csv(os.path.join(RESULT_PATH, "Table_4_DT_Distribution.csv"), index=False)


# ============================================================
# TABLE 6 — SDG DISTRIBUTION
# ============================================================
table6 = (
    df_master["dominant_sdg"]
    .value_counts()
    .reset_index()
)

table6.columns = ["SDG Domain", "Frequency"]
table6["Percentage (%)"] = (
    table6["Frequency"] / table6["Frequency"].sum() * 100
).round(1)

table6.to_csv(os.path.join(RESULT_PATH, "Table_6_SDG_Distribution.csv"), index=False)


# ============================================================
# TABLE 5 — DT × SDG COUNTS
# ============================================================
table5 = pd.crosstab(
    df_master["dominant_dt_theory"],
    df_master["dominant_sdg"]
)

table5.to_csv(os.path.join(RESULT_PATH, "Table_5_DT_SDG_Counts.csv"))


# ============================================================
# TABLE 7 — DT × SDG PERCENT
# ============================================================
table7 = (
    table5.div(table5.sum(axis=0), axis=1) * 100
).round(1)

table7.to_csv(os.path.join(RESULT_PATH, "Table_7_DT_SDG_Percent.csv"))


# ============================================================
# TABLE 8 — SDG MARGIN (CORRECT)
# ============================================================
table8 = (
    df_master.groupby("dominant_dt_theory")
    .agg(
        Mean_Margin=("sdg_margin", "mean"),
        Mean_Abs_Margin=("sdg_margin", lambda x: np.mean(np.abs(x)))
    )
    .reset_index()
    .rename(columns={"dominant_dt_theory": "DT Theory"})
    .round(4)
)

table8.to_csv(os.path.join(RESULT_PATH, "Table_8_SDG_Margin.csv"), index=False)


# ============================================================
# TABLE 9 — OUTCOME SIGNAL (IMPROVED)
# ============================================================

OUTCOME_TERMS = [
    "health","wellbeing","disease",
    "education","learning","skills",
    "inequality","equity",
    "governance","transparency","justice",
    "employment","wages"
]

def count_terms(text):
    if not isinstance(text, str):
        return 0
    text = text.lower()
    return sum(len(re.findall(r"\b"+t+r"\b", text)) for t in OUTCOME_TERMS)

df_tmp = df_documents.copy()

df_tmp["word_count"] = df_tmp["text"].str.split().str.len()
df_tmp["outcome_count"] = df_tmp["text"].apply(count_terms)

df_tmp["outcome_per_1000w"] = (
    df_tmp["outcome_count"] /
    df_tmp["word_count"].replace(0, np.nan) * 1000
).fillna(0)

df_tmp = df_tmp.merge(
    df_master[["paper_id","dominant_dt_theory"]],
    on="paper_id",
    how="inner"
)

table9 = (
    df_tmp.groupby("dominant_dt_theory")["outcome_per_1000w"]
    .agg(["mean","median"])
    .reset_index()
    .rename(columns={"dominant_dt_theory": "DT Theory"})
    .round(4)
)

table9.to_csv(os.path.join(RESULT_PATH, "Table_9_Outcome_Signal.csv"), index=False)


# ============================================================
# TABLE 10 — EFFECT SIZE (STRUCTURED, NOT RANDOM)
# ============================================================
table10 = pd.DataFrame({
    "Metric": ["SDG Margin"],
    "Std Dev": [df_master["sdg_margin"].std()],
    "Mean": [df_master["sdg_margin"].mean()]
}).round(4)

table10.to_csv(os.path.join(RESULT_PATH, "Table_10_Effect_Summary.csv"), index=False)


# ============================================================
# TABLE 12 — THEORY CENTRALITY (CLEAN)
# ============================================================

edges = []

for _, row in df_master.iterrows():
    edges.append(row["dominant_dt_theory"])

pairs = list(itertools.combinations(edges, 2))

G = nx.Graph()

for a, b in pairs:
    if a != b:
        if G.has_edge(a,b):
            G[a][b]["weight"] += 1
        else:
            G.add_edge(a,b,weight=1)

centrality = nx.eigenvector_centrality(G, max_iter=1000)

table12 = pd.DataFrame({
    "DT Theory": list(centrality.keys()),
    "Eigenvector Centrality": list(centrality.values())
}).sort_values("Eigenvector Centrality", ascending=False)

table12.to_csv(os.path.join(RESULT_PATH, "Table_12_Theory_Centrality.csv"), index=False)


# ============================================================
# FINAL CHECK
# ============================================================
print("\n✅ ALL TABLES CREATED:\n")

for f in sorted(os.listdir(RESULT_PATH)):
    if "Table" in f:
        print("-", f)

In [ ]:
# ============================================================
# KNOWLEDGE GRAPH (STRUCTURED, MULTI-LAYER)
# ============================================================

import os
import pandas as pd
import networkx as nx

# -----------------------------
# LOAD DATA
# -----------------------------
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

# -----------------------------
# VALIDATION
# -----------------------------
assert "paper_id" in df_master.columns
assert "dominant_dt_theory" in df_master.columns
assert "dominant_sdg" in df_master.columns

# ============================================================
# STEP 1 — BUILD TRIPLES
# ============================================================

triples = []

for _, row in df_master.iterrows():

    paper = row["paper_id"]
    theory = row["dominant_dt_theory"]
    sdg = row["dominant_sdg"]

    # paper → theory
    triples.append((paper, "uses", theory))

    # paper → SDG
    triples.append((paper, "targets", sdg))

    # theory → SDG
    triples.append((theory, "relates_to", sdg))

df_triples = pd.DataFrame(triples, columns=["source", "relation", "target"])

df_triples.to_csv(
    os.path.join(RESULT_PATH, "KG_triples.csv"),
    index=False
)

# ============================================================
# STEP 2 — BUILD GRAPH
# ============================================================

G = nx.DiGraph()

for _, row in df_triples.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        relation=row["relation"]
    )

print("Graph built:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# ============================================================
# STEP 3 — NODE TYPES (IMPORTANT FOR STRUCTURE)
# ============================================================

for node in G.nodes():
    if node.startswith("SDG"):
        G.nodes[node]["type"] = "sdg"
    elif node in df_master["dominant_dt_theory"].unique():
        G.nodes[node]["type"] = "theory"
    else:
        G.nodes[node]["type"] = "paper"

# ============================================================
# STEP 4 — CENTRALITY (THEORY LAYER ONLY)
# ============================================================

theories = df_master["dominant_dt_theory"].unique()

deg = nx.degree_centrality(G)
bet = nx.betweenness_centrality(G)

centrality = []

for t in theories:
    centrality.append({
        "DT Theory": t,
        "Degree": deg.get(t, 0),
        "Betweenness": bet.get(t, 0)
    })

df_centrality = pd.DataFrame(centrality).sort_values(
    "Betweenness",
    ascending=False
)

df_centrality.to_csv(
    os.path.join(RESULT_PATH, "KG_theory_centrality.csv"),
    index=False
)

# ============================================================
# STEP 5 — SDG–THEORY MATRIX (IMPORTANT TABLE)
# ============================================================

sdg_theory_matrix = pd.crosstab(
    df_master["dominant_dt_theory"],
    df_master["dominant_sdg"]
)

sdg_theory_matrix.to_csv(
    os.path.join(RESULT_PATH, "KG_sdg_theory_matrix.csv")
)

# ============================================================
# STEP 6 — SIMPLE VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

pos = nx.spring_layout(G, seed=42)

node_colors = []

for n in G.nodes():
    t = G.nodes[n]["type"]

    if t == "sdg":
        node_colors.append("red")
    elif t == "theory":
        node_colors.append("blue")
    else:
        node_colors.append("gray")

nx.draw(
    G,
    pos,
    node_color=node_colors,
    node_size=30,
    edge_color="lightgray",
    with_labels=False
)

plt.title("Knowledge Graph (Paper–Theory–SDG)")
plt.show()

# ============================================================
# STEP 7 — SAVE GRAPH
# ============================================================

nx.write_gexf(G, os.path.join(RESULT_PATH, "knowledge_graph.gexf"))

print("\nKnowledge graph saved:")
print("- KG_triples.csv")
print("- KG_theory_centrality.csv")
print("- KG_sdg_theory_matrix.csv")
print("- knowledge_graph.gexf")

In [ ]:
# ============================================================
# SDG NETWORKS (OPTIMIZED, FULL + SAMPLE SUPPORT)
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# -----------------------------
# LOAD DATA
# -----------------------------
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_documents.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# MODEL (LOAD ONCE)
# -----------------------------
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

# -----------------------------
# PARAMETERS
# -----------------------------
SIM_THRESHOLD = 0.70
SAMPLE_SIZE = 120   # set None for full corpus

# ============================================================
# STEP 1 — FILTER SDGs
# ============================================================

df_sdg3 = df[df["dominant_sdg"] == "SDG3"].reset_index(drop=True)
df_sdg16 = df[df["dominant_sdg"] == "SDG16"].reset_index(drop=True)

# optional sampling (for speed)
if SAMPLE_SIZE is not None:
    df_sdg3 = df_sdg3.sample(min(SAMPLE_SIZE, len(df_sdg3)), random_state=42)
    df_sdg16 = df_sdg16.sample(min(SAMPLE_SIZE, len(df_sdg16)), random_state=42)

# ============================================================
# STEP 2 — EMBEDDINGS (CACHE)
# ============================================================

def get_embeddings(texts, name):

    path = os.path.join(RESULT_PATH, f"embeddings_{name}.npy")

    if os.path.exists(path):
        return np.load(path)

    emb = model.encode(texts, normalize_embeddings=True)
    np.save(path, emb)

    return emb


emb_sdg3 = get_embeddings(df_sdg3["text"].tolist(), "sdg3")
emb_sdg16 = get_embeddings(df_sdg16["text"].tolist(), "sdg16")

# ============================================================
# STEP 3 — FAST GRAPH BUILDER
# ============================================================

def build_graph_fast(embeddings):

    sim = cosine_similarity(embeddings)

    G = nx.Graph()

    n = len(sim)

    for i in range(n):
        G.add_node(i)

    # vectorized edge creation
    rows, cols = np.where(sim >= SIM_THRESHOLD)

    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j, weight=sim[i, j])

    # largest connected component
    if G.number_of_nodes() > 0:
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    return G


G3 = build_graph_fast(emb_sdg3)
G16 = build_graph_fast(emb_sdg16)

# ============================================================
# STEP 4 — VISUALIZATION FUNCTION
# ============================================================

def plot_graph(G, title, color):

    plt.figure(figsize=(10,10))

    pos = nx.spring_layout(G, seed=42)

    nx.draw_networkx_nodes(G, pos, node_color=color, node_size=40, alpha=0.8)
    nx.draw_networkx_edges(G, pos, alpha=0.2)

    plt.title(title)
    plt.axis("off")
    plt.show()


# -----------------------------
# SDG3
# -----------------------------
plot_graph(G3, "SDG3 Network", "blue")

# -----------------------------
# SDG16
# -----------------------------
plot_graph(G16, "SDG16 Network", "red")

# ============================================================
# STEP 5 — COMBINED NETWORK
# ============================================================

G_combined = nx.Graph()

# SDG3
for n in G3.nodes():
    G_combined.add_node(f"A_{n}", sdg="SDG3")

for u, v in G3.edges():
    G_combined.add_edge(f"A_{u}", f"A_{v}")

# SDG16
for n in G16.nodes():
    G_combined.add_node(f"B_{n}", sdg="SDG16")

for u, v in G16.edges():
    G_combined.add_edge(f"B_{u}", f"B_{v}")

# plot
plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_combined, seed=42)

colors = [
    "blue" if G_combined.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_combined.nodes()
]

nx.draw_networkx_nodes(G_combined, pos, node_color=colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

plt.title("Combined Network: SDG3 (Blue) vs SDG16 (Red)")
plt.axis("off")
plt.show()

# ============================================================
# STEP 6 — CROSS-SDG NETWORK
# ============================================================

sim_cross = cosine_similarity(emb_sdg3, emb_sdg16)

G_cross = nx.Graph()

# nodes
for i in range(len(df_sdg3)):
    G_cross.add_node(f"A_{i}", sdg="SDG3")

for j in range(len(df_sdg16)):
    G_cross.add_node(f"B_{j}", sdg="SDG16")

# edges
rows, cols = np.where(sim_cross >= SIM_THRESHOLD)

for i, j in zip(rows, cols):
    G_cross.add_edge(f"A_{i}", f"B_{j}")

print("Cross edges:", G_cross.number_of_edges())

# plot
plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_cross, seed=42)

colors = [
    "blue" if G_cross.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_cross.nodes()
]

nx.draw_networkx_nodes(G_cross, pos, node_color=colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_cross, pos, alpha=0.2)

plt.title("Cross-SDG Network (SDG3 vs SDG16)")
plt.axis("off")
plt.show()

# ============================================================
# STEP 7 — SAVE
# ============================================================

nx.write_gexf(G3, os.path.join(RESULT_PATH, "SDG3_network.gexf"))
nx.write_gexf(G16, os.path.join(RESULT_PATH, "SDG16_network.gexf"))
nx.write_gexf(G_combined, os.path.join(RESULT_PATH, "Combined_network.gexf"))
nx.write_gexf(G_cross, os.path.join(RESULT_PATH, "Cross_SDG_network.gexf"))

print("\n✅ Networks saved")

In [ ]:
# ============================================================
# FULL NETWORK (NO SAMPLING, PAPER-CORRECT)
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# -----------------------------
# LOAD DATA
# -----------------------------
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_documents.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# MODEL
# -----------------------------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

SIM_THRESHOLD = 0.65

# ============================================================
# FILTER (FULL, NO SAMPLING)
# ============================================================

df_sdg3 = df[df["dominant_sdg"] == "SDG3"].reset_index(drop=True)
df_sdg16 = df[df["dominant_sdg"] == "SDG16"].reset_index(drop=True)

# ============================================================
# EMBEDDINGS (ONCE PER SDG)
# ============================================================

emb_sdg3 = model.encode(df_sdg3["text"].tolist(), normalize_embeddings=True)
emb_sdg16 = model.encode(df_sdg16["text"].tolist(), normalize_embeddings=True)

# ============================================================
# FAST GRAPH BUILDER (VECTOR + LCC)
# ============================================================

def build_graph(embeddings):

    sim = cosine_similarity(embeddings)
    G = nx.Graph()

    n = len(sim)

    for i in range(n):
        G.add_node(i)

    rows, cols = np.where(sim >= SIM_THRESHOLD)

    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    # IMPORTANT: LCC ONLY (as in your paper)
    if G.number_of_nodes() > 0:
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    return G


G3 = build_graph(emb_sdg3)
G16 = build_graph(emb_sdg16)

# ============================================================
# VISUALIZATION (FULL STRUCTURE)
# ============================================================

def plot(G, title, color):

    plt.figure(figsize=(10,10))
    pos = nx.spring_layout(G, seed=42)

    nx.draw_networkx_nodes(G, pos, node_color=color, node_size=30)
    nx.draw_networkx_edges(G, pos, alpha=0.2)

    plt.title(title)
    plt.axis("off")
    plt.show()


plot(G3, "SDG3 Network (Full)", "blue")
plot(G16, "SDG16 Network (Full)", "red")

In [ ]:
# how many SDG3 nodes connect to SDG16?

count = 0

for u, v in G_cross.edges():
    count += 1

print("Cross edges:", count)

In [ ]:
print("Total SDG3 nodes:", len(df_sdg3))
print("Total SDG16 nodes:", len(df_sdg16))

In [ ]:
for u, v in G_cross.edges():
    print(u, v)

In [ ]:
# assumes df_sdg3 and df_sdg16 are the exact subsets used to build G_cross
bridges = [("A_19","B_104"), ("A_52","B_32"), ("A_109","B_32"), ("A_109","B_99")]

rows = []
for a, b in bridges:
    i = int(a.split("_")[1])
    j = int(b.split("_")[1])

    rows.append({
        "A_node": a,
        "A_paper_id": df_sdg3.iloc[i]["paper_id"],
        "B_node": b,
        "B_paper_id": df_sdg16.iloc[j]["paper_id"]
    })

df_bridges = pd.DataFrame(rows)
df_bridges.to_csv(os.path.join(RESULT_PATH, "Table_Bridge_Pairs.csv"), index=False)
df_bridges

In [ ]:
def detect_sdg(file_name):

    fname = file_name.upper()

    if fname.startswith("SDGM"):
        return "SDGM"
    elif fname.startswith("SDG3"):
        return "SDG3"
    elif fname.startswith("SDG16"):
        return "SDG16"
    else:
        return "UNKNOWN"

In [ ]:
df_documents["sdg_label"] = df_documents["paper_id"].apply(detect_sdg)

In [ ]:
# The 'df' DataFrame needs to be updated with the 'sdg_label' column
# which was added to 'df_documents' in the previous step.
# We merge the 'sdg_label' column from the updated df_documents into df.
df = df.merge(df_documents[["paper_id", "sdg_label"]], on="paper_id", how="left")

df_sdg3  = df[df["sdg_label"] == "SDG3"]
df_sdg16 = df[df["sdg_label"] == "SDG16"]
df_sdgm  = df[df["sdg_label"] == "SDGM"]

In [ ]:
# embeddings
emb_3  = model.encode(df_sdg3["text"].tolist(), normalize_embeddings=True)
emb_16 = model.encode(df_sdg16["text"].tolist(), normalize_embeddings=True)
emb_m  = model.encode(df_sdgm["text"].tolist(), normalize_embeddings=True)

# similarities
sim_3_m  = cosine_similarity(emb_3, emb_m)
sim_16_m = cosine_similarity(emb_16, emb_m)

G_cross = nx.Graph()

# nodes
for i in range(len(df_sdg3)):
    G_cross.add_node(f"3_{i}", sdg="SDG3")

for i in range(len(df_sdgm)):
    G_cross.add_node(f"M_{i}", sdg="SDGM")

for i in range(len(df_sdg16)):
    G_cross.add_node(f"16_{i}", sdg="SDG16")

# edges: ONLY via SDGM
for i in range(len(df_sdg3)):
    for j in range(len(df_sdgm)):
        if sim_3_m[i, j] >= SIM_THRESHOLD:
            G_cross.add_edge(f"3_{i}", f"M_{j}")

for i in range(len(df_sdg16)):
    for j in range(len(df_sdgm)):
        if sim_16_m[i, j] >= SIM_THRESHOLD:
            G_cross.add_edge(f"16_{i}", f"M_{j}")

In [ ]:
print("\n=== SDG COUNTS ===")
print("SDG3:", len(df_sdg3))
print("SDG16:", len(df_sdg16))
print("SDGM (mixed):", len(df_sdgm))

In [ ]:
print("\n=== CROSS EDGES VIA SDGM ===")

edges_3_m = 0
edges_16_m = 0

for u, v in G_cross.edges():
    if u.startswith("3_") and v.startswith("M_"):
        edges_3_m += 1
    elif u.startswith("M_") and v.startswith("3_"):
        edges_3_m += 1
    elif u.startswith("16_") and v.startswith("M_"):
        edges_16_m += 1
    elif u.startswith("M_") and v.startswith("16_"):
        edges_16_m += 1

print("SDG3 → SDGM edges:", edges_3_m)
print("SDG16 → SDGM edges:", edges_16_m)
print("Total edges:", G_cross.number_of_edges())

In [ ]:
print("\n=== SAMPLE EDGES ===")

edges_list = list(G_cross.edges())

for e in edges_list[:20]:
    print(e)

In [ ]:
print("\n=== SDGM BRIDGE NODES ===")

bridge_nodes = set()

for u, v in G_cross.edges():
    if "M_" in u:
        bridge_nodes.add(u)
    if "M_" in v:
        bridge_nodes.add(v)

print("Number of SDGM bridge nodes:", len(bridge_nodes))

for b in list(bridge_nodes)[:20]:
    print(b)

In [ ]:
print("\n=== SDGM BRIDGE PAPERS ===")

for b in list(bridge_nodes)[:20]:
    idx = int(b.split("_")[1])
    print(b, "→", df_sdgm.iloc[idx]["paper_id"])

In [ ]:
N3 = len(df_sdg3)
N16 = len(df_sdg16)
NM = len(df_sdgm)

possible_3_m = N3 * NM
possible_16_m = N16 * NM

print("\n=== DENSITY ===")
print("SDG3–SDGM density:", edges_3_m / (possible_3_m + 1e-6))
print("SDG16–SDGM density:", edges_16_m / (possible_16_m + 1e-6))

In [ ]:
for u, v in G_cross.edges():
    print(u, v)

In [ ]:
# ============================================================
# FULL NETWORK BLOCK (COMBINED + CROSS + HIGHLIGHT)
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# -----------------------------
# LOAD DATA
# -----------------------------
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_documents.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# MODEL
# -----------------------------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

SIM_THRESHOLD = 0.65

# ============================================================
# SPLIT DATA
# ============================================================

df_sdg3 = df[df["dominant_sdg"] == "SDG3"].reset_index(drop=True)
df_sdg16 = df[df["dominant_sdg"] == "SDG16"].reset_index(drop=True)

# ============================================================
# EMBEDDINGS
# ============================================================

emb_3 = model.encode(df_sdg3["text"].tolist(), normalize_embeddings=True)
emb_16 = model.encode(df_sdg16["text"].tolist(), normalize_embeddings=True)

# ============================================================
# GRAPH BUILDER
# ============================================================

def build_graph(emb):

    sim = cosine_similarity(emb)
    G = nx.Graph()

    n = len(sim)

    for i in range(n):
        G.add_node(i)

    rows, cols = np.where(sim >= SIM_THRESHOLD)

    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    # largest connected component
    if G.number_of_nodes() > 0:
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    return G


G3 = build_graph(emb_3)
G16 = build_graph(emb_16)

# ============================================================
# CROSS SIMILARITY
# ============================================================

sim_cross = cosine_similarity(emb_3, emb_16)
rows_cross, cols_cross = np.where(sim_cross >= SIM_THRESHOLD)

print("Cross edges:", len(rows_cross))


# ============================================================
# COMBINED NETWORK (WITH CROSS EDGES)
# ============================================================

G_combined = nx.Graph()

# SDG3 nodes
for n in G3.nodes():
    G_combined.add_node(f"A_{n}", sdg="SDG3")

# SDG16 nodes
for n in G16.nodes():
    G_combined.add_node(f"B_{n}", sdg="SDG16")

# SDG3 internal edges
for u, v in G3.edges():
    G_combined.add_edge(f"A_{u}", f"A_{v}")

# SDG16 internal edges
for u, v in G16.edges():
    G_combined.add_edge(f"B_{u}", f"B_{v}")

# CROSS EDGES (CRITICAL)
cross_edges = []
for i, j in zip(rows_cross, cols_cross):
    edge = (f"A_{i}", f"B_{j}")
    G_combined.add_edge(*edge)
    cross_edges.append(edge)


# ============================================================
# VISUALIZE COMBINED NETWORK (WITH HIGHLIGHT)
# ============================================================

plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_combined, seed=42)

node_colors = [
    "blue" if G_combined.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_combined.nodes()
]

# nodes
nx.draw_networkx_nodes(G_combined, pos, node_color=node_colors, node_size=40, alpha=0.8)

# internal edges
nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

# highlight cross edges
nx.draw_networkx_edges(
    G_combined,
    pos,
    edgelist=cross_edges,
    width=3,
    edge_color="black"
)

plt.title("Combined Network: SDG3 (Blue) vs SDG16 (Red) with Cross Bridges")
plt.axis("off")
plt.show()


# ============================================================
# CROSS-SDG NETWORK (ONLY INTERACTIONS)
# ============================================================

G_cross = nx.Graph()

# ============================================================
# SAFE NODE CREATION (FIX)
# ============================================================

# SDG3 nodes
for i in range(len(df_sdg3)):
    G_combined.add_node(f"A_{i}", sdg="SDG3")

# SDG16 nodes
for j in range(len(df_sdg16)):
    G_combined.add_node(f"B_{j}", sdg="SDG16")

# edges
for i, j in zip(rows_cross, cols_cross):
    G_cross.add_edge(f"A_{i}", f"B_{j}")

# visualize
plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_cross, seed=42)

node_colors = [
    "blue" if G_cross.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_cross.nodes()
]

nx.draw_networkx_nodes(G_cross, pos, node_color=node_colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_cross, pos, width=2)

plt.title("Cross-SDG Network (SDG3 vs SDG16)")
plt.axis("off")
plt.show()


# ============================================================
# SAVE NETWORKS
# ============================================================

nx.write_gexf(G_combined, os.path.join(RESULT_PATH, "Combined_network.gexf"))
nx.write_gexf(G_cross, os.path.join(RESULT_PATH, "Cross_SDG_network.gexf"))

print("\n✅ Networks saved")

In [ ]:
import numpy as np

np.save(os.path.join(RESULT_PATH, "embeddings_sdg3.npy"), emb_3)
np.save(os.path.join(RESULT_PATH, "embeddings_sdg16.npy"), emb_16)

print("Embeddings saved")

In [ ]:
# ============================================================
# NETWORK BLOCK (NO EMBEDDING, LOAD FROM FILE)
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# LOAD DATA
# -----------------------------
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_documents.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# LOAD EMBEDDINGS (NO RECOMPUTE)
# -----------------------------
emb_3 = np.load(os.path.join(RESULT_PATH, "embeddings_sdg3.npy"))
emb_16 = np.load(os.path.join(RESULT_PATH, "embeddings_sdg16.npy"))

# -----------------------------
# FILTER DATA (must match embeddings)
# -----------------------------
df_sdg3 = df[df["dominant_sdg"] == "SDG3"].reset_index(drop=True)
df_sdg16 = df[df["dominant_sdg"] == "SDG16"].reset_index(drop=True)

SIM_THRESHOLD = 0.65

# ============================================================
# GRAPH BUILDER
# ============================================================

def build_graph(emb):

    sim = cosine_similarity(emb)
    G = nx.Graph()

    n = len(sim)

    for i in range(n):
        G.add_node(i)

    rows, cols = np.where(sim >= SIM_THRESHOLD)

    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    # LCC
    if G.number_of_nodes() > 0:
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    return G


G3 = build_graph(emb_3)
G16 = build_graph(emb_16)

# ============================================================
# CROSS SIMILARITY
# ============================================================

sim_cross = cosine_similarity(emb_3, emb_16)
rows_cross, cols_cross = np.where(sim_cross >= SIM_THRESHOLD)

print("Cross edges:", len(rows_cross))


# ============================================================
# COMBINED NETWORK (FIXED)
# ============================================================

G_combined = nx.Graph()

# SAFE NODE CREATION
for i in range(len(df_sdg3)):
    G_combined.add_node(f"A_{i}", sdg="SDG3")

for j in range(len(df_sdg16)):
    G_combined.add_node(f"B_{j}", sdg="SDG16")

# SDG3 internal edges
for u, v in G3.edges():
    G_combined.add_edge(f"A_{u}", f"A_{v}")

# SDG16 internal edges
for u, v in G16.edges():
    G_combined.add_edge(f"B_{u}", f"B_{v}")

# CROSS edges
cross_edges = []
for i, j in zip(rows_cross, cols_cross):
    edge = (f"A_{i}", f"B_{j}")
    G_combined.add_edge(*edge)
    cross_edges.append(edge)

# ============================================================
# VISUALIZE COMBINED
# ============================================================

plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_combined, seed=42)

node_colors = [
    "blue" if G_combined.nodes[n].get("sdg") == "SDG3" else "red"
    for n in G_combined.nodes()
]

# nodes
nx.draw_networkx_nodes(G_combined, pos, node_color=node_colors, node_size=40, alpha=0.8)

# internal edges
nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

# highlight cross edges
nx.draw_networkx_edges(
    G_combined,
    pos,
    edgelist=cross_edges,
    width=2,
    edge_color="black"
)

plt.title("Combined Network: SDG3 (Blue) vs SDG16 (Red)")
plt.axis("off")
plt.show()


# ============================================================
# CROSS-SDG NETWORK (ONLY INTERACTION)
# ============================================================

G_cross = nx.Graph()

# nodes
for i in range(len(df_sdg3)):
    G_cross.add_node(f"A_{i}", sdg="SDG3")

for j in range(len(df_sdg16)):
    G_cross.add_node(f"B_{j}", sdg="SDG16")

# edges
for i, j in zip(rows_cross, cols_cross):
    G_cross.add_edge(f"A_{i}", f"B_{j}")

# visualize
plt.figure(figsize=(12,10))
pos = nx.spring_layout(G_cross, seed=42)

node_colors = [
    "blue" if G_cross.nodes[n].get("sdg") == "SDG3" else "red"
    for n in G_cross.nodes()
]

nx.draw_networkx_nodes(G_cross, pos, node_color=node_colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_cross, pos, width=1.5)

plt.title("Cross-SDG Network (SDG3 vs SDG16)")
plt.axis("off")
plt.show()


# ============================================================
# SAVE
# ============================================================

nx.write_gexf(G_combined, os.path.join(RESULT_PATH, "Combined_network.gexf"))
nx.write_gexf(G_cross, os.path.join(RESULT_PATH, "Cross_SDG_network.gexf"))

print("\n✅ Networks saved")

In [ ]:
degree = dict(G_combined.degree())

core_nodes = [n for n, d in degree.items() if d > np.percentile(list(degree.values()), 75)]
core_sdg3 = sum(1 for n in core_nodes if G_combined.nodes[n]["sdg"] == "SDG3")
core_sdg16 = sum(1 for n in core_nodes if G_combined.nodes[n]["sdg"] == "SDG16")
print ("SDG3 ",core_sdg3)
print("SDG16 ",core_sdg16)

In [ ]:
degree = dict(G_combined.degree())

threshold = np.percentile(list(degree.values()), 75)

core = [n for n, d in degree.items() if d >= threshold]
periphery = [n for n in G_combined.nodes() if n not in core]

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_combined, seed=42)

nx.draw_networkx_nodes(G_combined, pos,
    nodelist=periphery,
    node_color="lightgray",
    node_size=20)

nx.draw_networkx_nodes(G_combined, pos,
    nodelist=core,
    node_color="black",
    node_size=60)

nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

plt.title("Core–Periphery Structure")
plt.axis("off")
plt.show()

In [ ]:
import seaborn as sns

deg = dict(G_combined.degree())

df_deg = pd.DataFrame({
    "node": list(deg.keys()),
    "degree": list(deg.values()),
    "sdg": [G_combined.nodes[n]["sdg"] for n in deg]
})

sns.kdeplot(data=df_deg, x="degree", hue="sdg")
plt.title("Degree Distribution by SDG")
plt.show()

In [ ]:
!pip install python-louvain
import community.community_louvain as community_louvain

partition = community_louvain.best_partition(G_combined)

colors = [partition[n] for n in G_combined.nodes()]

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_combined, seed=42)

nx.draw_networkx_nodes(G_combined, pos,
    node_color=colors,
    cmap=plt.cm.tab10,
    node_size=40)

nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

plt.title("Community Structure (Louvain)")
plt.axis("off")
plt.show()

In [ ]:
bet = nx.betweenness_centrality(G_combined)

top_bridge = sorted(bet, key=bet.get, reverse=True)[:20]

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_combined, seed=42)

nx.draw_networkx_nodes(G_combined, pos,
    nodelist=top_bridge,
    node_color="black",
    node_size=80)

nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

plt.title("Top Bridge Nodes (Betweenness)")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# REBUILD COMBINED GRAPH + COLORED BRIDGE NODES
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# LOAD DATA
# -----------------------------
df_documents = pd.read_csv(os.path.join(RESULT_PATH, "df_documents.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_documents.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# LOAD EMBEDDINGS (already saved)
# -----------------------------
emb_3 = np.load(os.path.join(RESULT_PATH, "embeddings_sdg3.npy"))
emb_16 = np.load(os.path.join(RESULT_PATH, "embeddings_sdg16.npy"))

df_sdg3 = df[df["dominant_sdg"] == "SDG3"].reset_index(drop=True)
df_sdg16 = df[df["dominant_sdg"] == "SDG16"].reset_index(drop=True)

SIM_THRESHOLD = 0.65

# ============================================================
# BUILD INTERNAL GRAPHS
# ============================================================

def build_graph(emb):
    sim = cosine_similarity(emb)
    G = nx.Graph()
    n = len(sim)

    for i in range(n):
        G.add_node(i)

    rows, cols = np.where(sim >= SIM_THRESHOLD)

    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    if G.number_of_nodes() > 0:
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    return G


G3 = build_graph(emb_3)
G16 = build_graph(emb_16)

# ============================================================
# CROSS EDGES
# ============================================================

sim_cross = cosine_similarity(emb_3, emb_16)
rows_cross, cols_cross = np.where(sim_cross >= SIM_THRESHOLD)

# ============================================================
# BUILD COMBINED GRAPH (SAFE)
# ============================================================

G_combined = nx.Graph()

# nodes
for i in range(len(df_sdg3)):
    G_combined.add_node(f"A_{i}", sdg="SDG3")

for j in range(len(df_sdg16)):
    G_combined.add_node(f"B_{j}", sdg="SDG16")

# internal edges
for u, v in G3.edges():
    G_combined.add_edge(f"A_{u}", f"A_{v}")

for u, v in G16.edges():
    G_combined.add_edge(f"B_{u}", f"B_{v}")

# cross edges
for i, j in zip(rows_cross, cols_cross):
    G_combined.add_edge(f"A_{i}", f"B_{j}")

# ============================================================
# BRIDGE NODES (BETWEENNESS)
# ============================================================

bet = nx.betweenness_centrality(G_combined)
top_bridge = sorted(bet, key=bet.get, reverse=True)[:20]

# ============================================================
# VISUALIZATION (COLORED)
# ============================================================

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_combined, seed=42)

# base nodes
node_colors = [
    "blue" if G_combined.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_combined.nodes()
]

nx.draw_networkx_nodes(
    G_combined,
    pos,
    node_color=node_colors,
    node_size=30,
    alpha=0.6
)

# edges
nx.draw_networkx_edges(G_combined, pos, alpha=0.1)

# highlight bridges
nx.draw_networkx_nodes(
    G_combined,
    pos,
    nodelist=top_bridge,
    node_color="black",
    node_size=120
)

plt.title("Top Bridge Nodes (Betweenness) — Colored")
plt.axis("off")
plt.show()

print("Top bridge nodes:", top_bridge[:10])

In [ ]:
# remove weak nodes (degree ≤ 1)
core_nodes = [n for n, d in G_combined.degree() if d > 1]

G_vis = G_combined.subgraph(core_nodes).copy()

In [ ]:
plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_vis, seed=42)

node_colors = [
    "blue" if G_vis.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_vis.nodes()
]

nx.draw_networkx_nodes(G_vis, pos, node_color=node_colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_vis, pos, alpha=0.15)

nx.draw_networkx_nodes(
    G_vis,
    pos,
    nodelist=[n for n in top_bridge if n in G_vis.nodes()],
    node_color="black",
    node_size=120
)

plt.title("Core Network (Filtered, No Peripheral Ring)")
plt.axis("off")
plt.show()

In [ ]:
G_vis = G_combined.copy()
G_vis.remove_nodes_from(list(nx.isolates(G_vis)))

In [ ]:
plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_vis, seed=42)

node_colors = [
    "blue" if G_vis.nodes[n]["sdg"] == "SDG3" else "red"
    for n in G_vis.nodes()
]

nx.draw_networkx_nodes(G_vis, pos, node_color=node_colors, node_size=40, alpha=0.8)
nx.draw_networkx_edges(G_vis, pos, alpha=0.15)

nx.draw_networkx_nodes(
    G_vis,
    pos,
    nodelist=[n for n in top_bridge if n in G_vis.nodes()],
    node_color="black",
    node_size=120
)

plt.title("Core Network (Filtered, No Peripheral Ring)")
plt.axis("off")
plt.show()

In [ ]:
df_deg.groupby("sdg")["degree"].mean().plot(kind="bar")
plt.title("Average Degree by SDG")
plt.ylabel("Mean Degree")
plt.show()

In [ ]:
def density(G):
    n = G.number_of_nodes()
    e = G.number_of_edges()
    return e / (n*(n-1)/2)

print("SDG3 density:", density(G3))
print("SDG16 density:", density(G16))

In [ ]:
modularity = community_louvain.modularity(partition, G_combined)
print("Modularity:", modularity)


In [ ]:
import community.community_louvain as community_louvain
import numpy as np

# SDG node lists
nodes_sdg3  = [n for n in G_combined.nodes() if G_combined.nodes[n].get("sdg") == "SDG3"]
nodes_sdg16 = [n for n in G_combined.nodes() if G_combined.nodes[n].get("sdg") == "SDG16"]

# induced subgraphs
G3_sub  = G_combined.subgraph(nodes_sdg3).copy()
G16_sub = G_combined.subgraph(nodes_sdg16).copy()

# partitions on subgraphs
part3  = community_louvain.best_partition(G3_sub)  if G3_sub.number_of_edges()  > 0 else {}
part16 = community_louvain.best_partition(G16_sub) if G16_sub.number_of_edges() > 0 else {}

# modularity per SDG
mod3  = community_louvain.modularity(part3,  G3_sub)  if part3  else np.nan
mod16 = community_louvain.modularity(part16, G16_sub) if part16 else np.nan

print("Modularity SDG3 :", mod3)
print("Modularity SDG16:", mod16)

df_mod_sdg = pd.DataFrame({
    "SDG": ["SDG3","SDG16"],
    "Modularity": [mod3, mod16]
})
df_mod_sdg.to_csv(os.path.join(RESULT_PATH, "Table_Modularity_per_SDG.csv"), index=False)
df_mod_sdg

In [ ]:
import pandas as pd
from collections import Counter
from math import log

# community labels from combined graph
comm = partition  # dict: node -> community

# build dataframe
df_comm = pd.DataFrame({
    "node": list(G_combined.nodes()),
    "community": [comm[n] for n in G_combined.nodes()],
    "sdg": [G_combined.nodes[n].get("sdg") for n in G_combined.nodes()]
})

# purity: for each community, majority SDG share
purities = []
for c, g in df_comm.groupby("community"):
    counts = g["sdg"].value_counts(normalize=True)
    purities.append(counts.iloc[0])

purity = float(np.mean(purities))
print("Purity:", purity)

# NMI (simple implementation)
def entropy(labels):
    c = Counter(labels)
    n = sum(c.values())
    return -sum((v/n) * log(v/n + 1e-12) for v in c.values())

def nmi(x, y):
    # x: communities, y: sdg
    Hx = entropy(x)
    Hy = entropy(y)
    I = 0.0
    cx = Counter(x)
    cy = Counter(y)
    n = len(x)
    for i in cx:
        for j in cy:
            pij = sum(1 for a,b in zip(x,y) if a==i and b==j) / n
            if pij > 0:
                pi = cx[i]/n
                pj = cy[j]/n
                I += pij * log(pij/(pi*pj) + 1e-12)
    return I / (0.5*(Hx+Hy) + 1e-12)

nmi_val = nmi(df_comm["community"].tolist(), df_comm["sdg"].tolist())
print("NMI:", nmi_val)

df_align = pd.DataFrame({
    "Metric": ["Purity","NMI"],
    "Value": [purity, nmi_val]
})
df_align.to_csv(os.path.join(RESULT_PATH, "Table_Community_SDG_Alignment.csv"), index=False)
df_align

In [ ]:
import numpy as np
from scipy.stats import spearmanr

# --- (A) Edge density (cross) ---
N3  = len([n for n in G_combined.nodes() if G_combined.nodes[n]["sdg"]=="SDG3"])
N16 = len([n for n in G_combined.nodes() if G_combined.nodes[n]["sdg"]=="SDG16"])

E_cross = len(cross_edges)
edge_density = E_cross / (N3 * N16 + 1e-12)

# --- (B) Dependency (bridge vs non-bridge centrality) ---
bet = nx.betweenness_centrality(G_combined)

bridge_nodes = set()
for u, v in cross_edges:
    bridge_nodes.add(u); bridge_nodes.add(v)

bridge_c = [bet[n] for n in bridge_nodes if n in bet]
non_c    = [bet[n] for n in bet if n not in bridge_nodes]

dep = (np.mean(bridge_c) / (np.mean(non_c) + 1e-12)) if (bridge_c and non_c) else 0.0

# --- (C) Centrality overlap (top-k degree ranks across SDGs) ---
deg = dict(G_combined.degree())

deg3  = {n:deg[n] for n in G_combined.nodes() if G_combined.nodes[n]["sdg"]=="SDG3"}
deg16 = {n:deg[n] for n in G_combined.nodes() if G_combined.nodes[n]["sdg"]=="SDG16"}

# take top-k (same k)
k = min(50, len(deg3), len(deg16))
top3  = sorted(deg3,  key=deg3.get,  reverse=True)[:k]
top16 = sorted(deg16, key=deg16.get, reverse=True)[:k]

vals3  = [deg3[n]  for n in top3]
vals16 = [deg16[n] for n in top16]

rho, _ = spearmanr(vals3, vals16)
centrality_overlap = 0 if np.isnan(rho) else float(rho)

# --- (D) SCI ---
alpha, beta, gamma = 0.4, 0.4, 0.2
SCI = alpha*edge_density + beta*dep + gamma*centrality_overlap

df_sci = pd.DataFrame([{
    "edge_density": edge_density,
    "dependency": dep,
    "centrality_overlap": centrality_overlap,
    "SCI": SCI
}])

df_sci.to_csv(os.path.join(RESULT_PATH, "Table_SCI.csv"), index=False)

print("Edge density:", edge_density)
print("Dependency:", dep)
print("Centrality overlap:", centrality_overlap)
print("SCI:", SCI)

df_sci

In [ ]:
# ============================================================
# THEORY NETWORK (NODES = THEORIES, EDGES = PAPERS)
# ============================================================

import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# LOAD THEORY MATRIX
# -----------------------------
df_theory = pd.read_csv(os.path.join(RESULT_PATH, "dt_theory_similarity_matrix.csv"))

# load SDG labels
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_theory.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# PARAMETERS
# -----------------------------
THEORY_THRESHOLD = 0.30

theory_cols = [
    c for c in df.columns
    if c not in ["paper_id", "dominant_dt_theory", "dominant_sdg"]
]

# ============================================================
# FUNCTION — BUILD THEORY GRAPH
# ============================================================

def build_theory_graph(df_subset):

    edges = []

    for _, row in df_subset.iterrows():

        # active theories in this paper
        active = [
            t for t in theory_cols
            if row[t] >= THEORY_THRESHOLD
        ]

        # create pairwise edges
        for i in range(len(active)):
            for j in range(i+1, len(active)):
                edges.append((active[i], active[j]))

    # count edge weights
    df_edges = (
        pd.DataFrame(edges, columns=["A","B"])
        .value_counts()
        .reset_index(name="weight")
    )

    # build graph
    G = nx.Graph()

    for _, r in df_edges.iterrows():
        G.add_edge(r["A"], r["B"], weight=r["weight"])

    return G, df_edges


# ============================================================
# BUILD SDG-SPECIFIC THEORY NETWORKS
# ============================================================

df_sdg3 = df[df["dominant_sdg"] == "SDG3"]
df_sdg16 = df[df["dominant_sdg"] == "SDG16"]

G3_theory, edges3 = build_theory_graph(df_sdg3)
G16_theory, edges16 = build_theory_graph(df_sdg16)

# ============================================================
# VISUALIZATION
# ============================================================

def plot_theory_graph(G, title):

    plt.figure(figsize=(8,8))

    pos = nx.spring_layout(G, seed=42)

    weights = [G[u][v]["weight"] for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_size=1500)
    nx.draw_networkx_labels(G, pos)

    nx.draw_networkx_edges(
        G, pos,
        width=[w/10 for w in weights],
        alpha=0.6
    )

    plt.title(title)
    plt.axis("off")
    plt.show()


# -----------------------------
# SDG3 THEORY NETWORK
# -----------------------------
plot_theory_graph(G3_theory, "DT Theory Network — SDG3")

# -----------------------------
# SDG16 THEORY NETWORK
# -----------------------------
plot_theory_graph(G16_theory, "DT Theory Network — SDG16")


# ============================================================
# SAVE
# ============================================================

nx.write_gexf(G3_theory, os.path.join(RESULT_PATH, "Theory_SDG3.gexf"))
nx.write_gexf(G16_theory, os.path.join(RESULT_PATH, "Theory_SDG16.gexf"))

edges3.to_csv(os.path.join(RESULT_PATH, "Theory_edges_SDG3.csv"), index=False)
edges16.to_csv(os.path.join(RESULT_PATH, "Theory_edges_SDG16.csv"), index=False)

print("✅ Theory networks created")

In [ ]:
# ============================================================
# THEORY NETWORK (PAPER-LEVEL, NOT AGGREGATED)
# ============================================================

import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# LOAD DATA
# -----------------------------
df_theory = pd.read_csv(os.path.join(RESULT_PATH, "dt_theory_similarity_matrix.csv"))
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

df = df_theory.merge(
    df_master[["paper_id", "dominant_sdg"]],
    on="paper_id",
    how="inner"
)

# -----------------------------
# PARAMETERS
# -----------------------------
THRESHOLD = 0.30

theory_cols = [
    c for c in df.columns
    if c not in ["paper_id", "dominant_dt_theory", "dominant_sdg"]
]

# ============================================================
# BUILD PAPER-LEVEL GRAPH
# ============================================================

G = nx.MultiGraph()   # IMPORTANT: MultiGraph (keeps all papers)

for _, row in df.iterrows():

    paper = row["paper_id"]

    active = [
        t for t in theory_cols
        if row[t] >= THRESHOLD
    ]

    # create edges per paper
    for i in range(len(active)):
        for j in range(i+1, len(active)):

            G.add_edge(
                active[i],
                active[j],
                paper=paper
            )

print("Nodes:", G.number_of_nodes())
print("Edges (papers):", G.number_of_edges())

In [ ]:
# ============================================================
# VISUALIZE (EDGE DENSITY = PAPER COUNT)
# ============================================================

# convert to weighted simple graph for plotting
G_plot = nx.Graph()

for u, v, data in G.edges(data=True):
    if G_plot.has_edge(u, v):
        G_plot[u][v]["weight"] += 1
    else:
        G_plot.add_edge(u, v, weight=1)

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G_plot, seed=42)

weights = [G_plot[u][v]["weight"] for u, v in G_plot.edges()]

nx.draw_networkx_nodes(G_plot, pos, node_size=1500)
nx.draw_networkx_labels(G_plot, pos)

nx.draw_networkx_edges(
    G_plot,
    pos,
    width=[w/20 for w in weights],
    alpha=0.6
)

plt.title("DT Theory Network (Edges = Papers)")
plt.axis("off")
plt.show()

In [ ]:
# find papers linking TAM and TOE
edges = G.get_edge_data("TAM", "TOE")

papers = [edges[k]["paper"] for k in edges]

print(papers[:20])

In [ ]:
# ============================================================
# SDG–THEORY GRAPH (EDGES = PAPERS, FULL DETAIL)
# ============================================================

import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# LOAD DATA
# -----------------------------
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))

# -----------------------------
# BUILD GRAPH
# -----------------------------
G = nx.MultiGraph()   # IMPORTANT: MultiGraph (keeps all papers)

for _, row in df_master.iterrows():

    paper = row["paper_id"]
    theory = row["dominant_dt_theory"]
    sdg = row["dominant_sdg"]

    # create nodes
    G.add_node(theory, type="theory")
    G.add_node(sdg, type="sdg")

    # create edge = paper
    G.add_edge(
        theory,
        sdg,
        paper=paper
    )

print("Nodes:", G.number_of_nodes())
print("Edges (papers):", G.number_of_edges())

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

plt.figure(figsize=(10,8))

pos = nx.spring_layout(G, seed=42)

# color nodes
node_colors = []

for n in G.nodes():
    if G.nodes[n]["type"] == "sdg":
        node_colors.append("red")
    else:
        node_colors.append("blue")

# draw nodes
nx.draw_networkx_nodes(
    G,
    pos,
    node_color=node_colors,
    node_size=1500
)

# labels
nx.draw_networkx_labels(G, pos)

# edges (each paper is one edge)
nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.05   # low alpha → shows density of papers
)

plt.title("SDG–DT Theory Network (Edges = Individual Papers)")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# ADD SDG–SDG CONNECTION BASED ON SHARED THEORIES
# ============================================================

theories_sdg3 = set(
    df_master[df_master["dominant_sdg"] == "SDG3"]["dominant_dt_theory"]
)

theories_sdg16 = set(
    df_master[df_master["dominant_sdg"] == "SDG16"]["dominant_dt_theory"]
)

shared = theories_sdg3.intersection(theories_sdg16)

print("Shared theories:", shared)

# weight = number of shared theories
G.add_edge("SDG3", "SDG16", weight=len(shared))

In [ ]:
# count papers that connect both SDGs
df_mixed = df_master.groupby("paper_id")["dominant_sdg"].nunique()
shared_papers = df_mixed[df_mixed > 1].count()

G.add_edge("SDG3", "SDG16", weight=shared_papers)

In [ ]:
# ============================================================
# TRUE GRAPH (THEORY + PAPER + SDG)
# ============================================================

G = nx.Graph()

for _, row in df_master.iterrows():

    paper = row["paper_id"]
    theory = row["dominant_dt_theory"]
    sdg = row["dominant_sdg"]

    # add nodes
    G.add_node(paper, type="paper")
    G.add_node(theory, type="theory")
    G.add_node(sdg, type="sdg")

    # connect paper to both
    G.add_edge(paper, theory)
    G.add_edge(paper, sdg)

In [ ]:
# ============================================================
# CREATE SDG–SDG EDGES FROM SHARED PAPERS
# ============================================================

from collections import defaultdict

paper_sdgs = defaultdict(set)

for _, row in df_master.iterrows():
    paper_sdgs[row["paper_id"]].add(row["dominant_sdg"])

G_sdg = nx.Graph()

for paper, sdgs in paper_sdgs.items():
    sdgs = list(sdgs)

    for i in range(len(sdgs)):
        for j in range(i+1, len(sdgs)):
            G_sdg.add_edge(sdgs[i], sdgs[j])

In [ ]:
# ============================================================
# TRUE MULTI-LAYER GRAPH (WITH VISUALIZATION)
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,10))

# layout (important: this graph is large)
pos = nx.spring_layout(G, seed=42, k=0.15)

# -----------------------------
# NODE COLORS
# -----------------------------
node_colors = []

for n in G.nodes():
    t = G.nodes[n]["type"]

    if t == "sdg":
        node_colors.append("red")
    elif t == "theory":
        node_colors.append("blue")
    else:
        node_colors.append("lightgray")  # papers

# -----------------------------
# DRAW NODES
# -----------------------------
nx.draw_networkx_nodes(
    G,
    pos,
    node_color=node_colors,
    node_size=20,
    alpha=0.7
)

# -----------------------------
# DRAW EDGES (papers)
# -----------------------------
nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.05
)

plt.title("Multi-layer Graph (Paper–Theory–SDG)")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

# keep only SDG + theory nodes
nodes_keep = [
    n for n in G.nodes()
    if G.nodes[n]["type"] in ["sdg", "theory"]
]

G_small = G.subgraph(nodes_keep)

pos = nx.spring_layout(G_small, seed=42)

node_colors = [
    "red" if G_small.nodes[n]["type"] == "sdg" else "blue"
    for n in G_small.nodes()
]

nx.draw_networkx_nodes(G_small, pos, node_color=node_colors, node_size=1500)
nx.draw_networkx_labels(G_small, pos)

nx.draw_networkx_edges(G_small, pos, alpha=0.3)

plt.title("SDG–Theory Network (Paper Connections Hidden)")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# 3-LAYER GRAPH (SDG – PAPER – THEORY, BRIDGE PAPERS ONLY)
# ============================================================

import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# LOAD DATA
# -----------------------------
df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))
df_theory = pd.read_csv(os.path.join(RESULT_PATH, "dt_theory_similarity_matrix.csv"))

df = df_master.merge(df_theory, on="paper_id", how="inner")

# -----------------------------
# THEORY COLUMNS (SAFE)
# -----------------------------
theory_cols = [
    "TAM_y", "UTAUT_y", "STS_y", "TOE_y", "RBV_y",
    "Dynamic_Capabilities_y", "IDT_y",
    "Stakeholder_Theory_y", "Just_Digital_y"
]

# force numeric (critical fix)
for t in theory_cols:
    df[t] = pd.to_numeric(df[t], errors="coerce")

# -----------------------------
# PARAMETERS
# -----------------------------
THEORY_THRESHOLD = 0.30
TOP_K = 25

# -----------------------------
# SELECT BRIDGE PAPERS
# -----------------------------
# use SDGM (clean and reliable)
bridge_papers = df[df["paper_id"].str.startswith("SDGM")]["paper_id"].tolist()[:TOP_K]

print("Bridge papers:", len(bridge_papers))

# -----------------------------
# BUILD GRAPH
# -----------------------------
G3L = nx.Graph()

for _, row in df[df["paper_id"].isin(bridge_papers)].iterrows():

    paper = row["paper_id"]
    sdg = row["dominant_sdg"]

    # nodes
    G3L.add_node(paper, type="paper")
    G3L.add_node(sdg, type="sdg")

    # edge: paper ↔ SDG
    G3L.add_edge(paper, sdg)

    # edges: paper ↔ theories
    for t in theory_cols:
        val = row[t]

        if pd.notna(val) and val >= THEORY_THRESHOLD:
            G3L.add_node(t.replace("_y", ""), type="theory") # Remove _y for node name
            G3L.add_edge(paper, t.replace("_y", "")) # Remove _y for edge name

print("Nodes:", G3L.number_of_nodes())
print("Edges:", G3L.number_of_edges())

# -----------------------------
# CLEAN 3-COLUMN LAYOUT
# -----------------------------
pos = {}

sdg_nodes = [n for n in G3L.nodes() if G3L.nodes[n]["type"] == "sdg"]
paper_nodes = [n for n in G3L.nodes() if G3L.nodes[n]["type"] == "paper"]
theory_nodes = [n for n in G3L.nodes() if G3L.nodes[n]["type"] == "theory"]

# fixed columns
def place(nodes, x):
    ys = np.linspace(0, 1, len(nodes)) if len(nodes) > 1 else [0.5]
    return {n: (x, y) for n, y in zip(nodes, ys)}

pos.update(place(sdg_nodes, 0))     # left
pos.update(place(paper_nodes, 1))   # center
pos.update(place(theory_nodes, 2))  # right

# -----------------------------
# COLORS
# -----------------------------
node_colors = []

for n in G3L.nodes():
    t = G3L.nodes[n]["type"]

    if t == "sdg":
        node_colors.append("red")
    elif t == "theory":
        node_colors.append("blue")
    else:
        node_colors.append("black")

# -----------------------------
# DRAW
# -----------------------------
plt.figure(figsize=(12,8))

nx.draw_networkx_nodes(
    G3L,
    pos,
    node_color=node_colors,
    node_size=300,
    alpha=0.9
)

nx.draw_networkx_edges(
    G3L,
    pos,
    alpha=0.4
)

# label only SDG + theory (clean)
labels = {
    n: n for n in G3L.nodes()
    if G3L.nodes[n]["type"] != "paper"
}

nx.draw_networkx_labels(G3L, pos, labels=labels, font_size=9)

plt.title("3-Layer Bridge Structure (SDG – Paper – Theory)")
plt.axis("off")
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np

df_master = pd.read_csv(os.path.join(RESULT_PATH, "df_master.csv"))
df_theory = pd.read_csv(os.path.join(RESULT_PATH, "dt_theory_similarity_matrix.csv"))

df = df_master.merge(df_theory, on="paper_id", how="inner")

# robust detection of theory columns (numeric only)
theory_cols = [
    c for c in df.columns
    if c not in ["paper_id", "dominant_dt_theory", "dominant_sdg"]
    and pd.api.types.is_numeric_dtype(df[c])
]

# enforce numeric (safety)
for t in theory_cols:
    df[t] = pd.to_numeric(df[t], errors="coerce")

print("Theory columns:", theory_cols)

In [ ]:
bridge_df = df[df["paper_id"].str.startswith("SDGM")].copy()

print("Bridge papers:", len(bridge_df))

In [ ]:
THRESHOLD = 0.30

rows = []

for _, r in bridge_df.iterrows():
    active_theories = [
        t for t in theory_cols
        if pd.notna(r[t]) and r[t] >= THRESHOLD
    ]

    rows.append({
        "paper_id": r["paper_id"],
        "dominant_sdg": r["dominant_sdg"],
        "theories": ", ".join(active_theories)
    })

df_bridge_table = pd.DataFrame(rows)

df_bridge_table.to_csv(
    os.path.join(RESULT_PATH, "Table_Bridge_Papers_Theories_SDG.csv"),
    index=False
)

df_bridge_table.head(10)

In [ ]:
bridge_df["bridge_strength"] = bridge_df[theory_cols].sum(axis=1)

top5 = bridge_df.sort_values("bridge_strength", ascending=False).head(5)

df_top5 = top5[["paper_id", "dominant_sdg", "bridge_strength"]]

df_top5.to_csv(
    os.path.join(RESULT_PATH, "Table_Top5_Bridge_Papers.csv"),
    index=False
)

df_top5

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G3L = nx.Graph()

top_ids = set(df_top5["paper_id"])

for _, r in bridge_df.iterrows():
    paper = r["paper_id"]
    sdg = r["dominant_sdg"]

    G3L.add_node(paper, type="paper")
    G3L.add_node(sdg, type="sdg")
    G3L.add_edge(paper, sdg)

    for t in theory_cols:
        if pd.notna(r[t]) and r[t] >= THRESHOLD:
            G3L.add_node(t, type="theory")
            G3L.add_edge(paper, t)

# layout (3 columns)
pos = {}
sdg_nodes = [n for n in G3L if G3L.nodes[n]["type"]=="sdg"]
paper_nodes = [n for n in G3L if G3L.nodes[n]["type"]=="paper"]
theory_nodes = [n for n in G3L if G3L.nodes[n]["type"]=="theory"]

def place(nodes, x):
    ys = np.linspace(0, 1, len(nodes)) if len(nodes)>1 else [0.5]
    return {n:(x,y) for n,y in zip(nodes, ys)}

pos.update(place(sdg_nodes, 0))
pos.update(place(paper_nodes, 1))
pos.update(place(theory_nodes, 2))

plt.figure(figsize=(12,8))

# base nodes
base_colors = []
for n in G3L.nodes():
    t = G3L.nodes[n]["type"]
    if t=="sdg": base_colors.append("red")
    elif t=="theory": base_colors.append("blue")
    else: base_colors.append("lightgray")

nx.draw_networkx_nodes(G3L, pos, node_color=base_colors, node_size=250, alpha=0.8)
nx.draw_networkx_edges(G3L, pos, alpha=0.3)

# highlight top 5 papers
nx.draw_networkx_nodes(
    G3L, pos,
    nodelist=list(top_ids),
    node_color="black",
    node_size=500
)

# labels (SDG + theory only)
labels = {n:n for n in G3L if G3L.nodes[n]["type"]!="paper"}
nx.draw_networkx_labels(G3L, pos, labels=labels, font_size=9)

plt.title("Top 5 Bridge Papers Highlighted")
plt.axis("off")
plt.show()

In [ ]:
from collections import Counter

counter = Counter()

for _, r in bridge_df.iterrows():
    for t in theory_cols:
        if pd.notna(r[t]) and r[t] >= THRESHOLD:
            counter[t] += 1

df_freq = pd.DataFrame({
    "theory": list(counter.keys()),
    "count": list(counter.values())
}).sort_values("count", ascending=False)

df_freq["percent"] = (df_freq["count"] / df_freq["count"].sum() * 100).round(2)

df_freq.to_csv(
    os.path.join(RESULT_PATH, "Table_Theory_Frequency_Bridges.csv"),
    index=False
)

df_freq

In [ ]:
# ============================================================
# 3D MULTI-LAYER GRAPH (SDG–PAPER–THEORY)
# ============================================================

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

pos = {}

# layers
z_sdg = 0
z_paper = 1
z_theory = 2

# assign positions
def place(nodes, z):
    xs = np.linspace(0, 1, len(nodes)) if len(nodes) > 1 else [0.5]
    ys = np.linspace(0, 1, len(nodes)) if len(nodes) > 1 else [0.5]
    return {n: (x, y, z) for n, x, y in zip(nodes, xs, ys)}

pos.update(place(sdg_nodes, z_sdg))
pos.update(place(paper_nodes, z_paper))
pos.update(place(theory_nodes, z_theory))

# draw nodes
for n in G3L.nodes():
    x, y, z = pos[n]
    t = G3L.nodes[n]["type"]

    if t == "sdg":
        ax.scatter(x, y, z, color="red", s=100)
    elif t == "theory":
        ax.scatter(x, y, z, color="blue", s=100)
    else:
        ax.scatter(x, y, z, color="black", s=20)

# draw edges
for u, v in G3L.edges():
    x = [pos[u][0], pos[v][0]]
    y = [pos[u][1], pos[v][1]]
    z = [pos[u][2], pos[v][2]]
    ax.plot(x, y, z, color="gray", alpha=0.3)

ax.set_title("3D Multi-layer Bridge Structure")
ax.set_axis_off()

plt.show()